# M44 lab solution: GraphRAG vs RAPTOR vs hybrid

**Brief.** GraphRAG vs RAPTOR vs hybrid: run the same global and multi-hop question set through each and write a one-page recommendation including indexing cost.

The corpus is the 20 documents of the M43 lab (13 handbook files and seven team and incident records); the questions are the M42 global set (10) and the M43 multi-hop set (5), copied into `../data/`.

1. **GraphRAG**, in the shape of Microsoft GraphRAG (lesson 44.3): extract entities and relationships per chunk, group the graph into communities (Louvain), write a report per community. **Global search** maps a question over the community reports and reduces the scored points into one answer; **local search** starts at the entities nearest the question and reads their relationships and source chunks. As GraphRAG is used, global questions go to global search and multi-hop questions to local search.
2. **RAPTOR** (M42): a tree of cluster summaries over the chunks, searched as a collapsed tree.
3. **Hybrid**: RAPTOR's top nodes and GraphRAG's local context in one prompt, for every question.

Indexing cost is measured per system in model tokens (`UsageMetadataCallbackHandler`) and wall-clock seconds. An LLM judge scores every answer 1 to 5 against its reference. The last cell writes the recommendation from the measured numbers.

## Setup

In [ ]:
import sys
import time
from collections import defaultdict
from pathlib import Path

import networkx as nx
import numpy as np
from dotenv import load_dotenv
from langchain_core.callbacks import UsageMetadataCallbackHandler
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pydantic import BaseModel, Field
from sklearn.mixture import GaussianMixture

load_dotenv()
sys.path.insert(0, str(Path("../../../../src").resolve()))
from helpers.utils import get_embeddings, get_experientiallabs_llm  # noqa: E402
from lab_kit import load_handbook, load_jsonl  # noqa: E402

DATA = Path("../data")
llm = get_experientiallabs_llm()
embeddings = get_embeddings()
MODEL = llm.model_name

pages = load_handbook(DATA / "larkspur" / "handbook") + load_handbook(DATA / "extra_docs")
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = [Document(page_content=c, metadata={"source": p["source"], "chunk_id": i})
          for i, (p, c) in enumerate((p, c) for p in pages for c in splitter.split_text(f"{p['title']}\n\n{p['text']}"))]
QUESTIONS = ([{**q, "set": "global"} for q in load_jsonl(DATA / "global_questions.jsonl")]
             + [{**q, "set": "multi_hop"} for q in load_jsonl(DATA / "multihop.jsonl")])
print("model:", MODEL, "|", len(pages), "documents,", len(chunks), "chunks |", len(QUESTIONS), "questions")


def tokens(cb: UsageMetadataCallbackHandler) -> int:
    return sum(u.get("total_tokens", 0) for u in cb.usage_metadata.values())


def answer_from(context: str, question: str) -> str:
    return llm.invoke([SystemMessage("Answer from the context only. Cover every relevant point it gives; if it does "
                                     "not answer, say what is missing.\n\nContext:\n" + context),
                       HumanMessage(question)]).content


INDEX = {}  # system -> {"seconds", "llm_tokens"}

## GraphRAG: index

Extraction uses the same entity and relationship schema as the M42 LightRAG build. Communities come from `networkx.community.louvain_communities` (seed fixed); each community with at least two entities gets a report: a title and a summary of its entities and relationships, written by the model.

In [ ]:
class Entity(BaseModel):
    name: str = Field(description="Canonical name, capitalized so it can be merged across chunks")
    type: str = Field(description="Entity type, e.g. plan, policy, region, team, person, service, incident")
    description: str = Field(description="Concise description based ONLY on the text")


class Relationship(BaseModel):
    source: str
    target: str
    description: str = Field(description="How the two entities are related, based on the text")


class GraphExtraction(BaseModel):
    entities: list[Entity] = Field(default_factory=list)
    relationships: list[Relationship] = Field(default_factory=list)


class Report(BaseModel):
    title: str
    summary: str = Field(description="What this group of entities covers, keeping every number, limit and name")


EXTRACT = ("Extract a knowledge graph from the text: the key entities and the relationships between them. Use only "
           "the text and keep names consistent.\n\nText:\n{text}")

graph_cb = UsageMetadataCallbackHandler()
start = time.perf_counter()
kg, entity_chunks = nx.Graph(), defaultdict(set)
extractor = llm.with_structured_output(GraphExtraction)
for d in chunks:
    cid = d.metadata["chunk_id"]
    try:
        ex = extractor.invoke(EXTRACT.format(text=d.page_content), config={"callbacks": [graph_cb]})
    except Exception as exc:  # noqa: BLE001
        print("skipped chunk", cid, exc)
        continue
    for e in ex.entities:
        key = e.name.strip().lower()
        if key:
            if key not in kg:
                kg.add_node(key, name=e.name, type=e.type, descriptions=[])
            kg.nodes[key]["descriptions"].append(e.description)
            entity_chunks[key].add(cid)
    for r in ex.relationships:
        s, t = r.source.strip().lower(), r.target.strip().lower()
        if s and t and s != t:
            for key, raw in ((s, r.source), (t, r.target)):
                if key not in kg:
                    kg.add_node(key, name=raw, type="unknown", descriptions=[])
                entity_chunks[key].add(cid)
            if kg.has_edge(s, t):
                kg.edges[s, t]["descriptions"].append(r.description)
                kg.edges[s, t]["chunk_ids"].add(cid)
            else:
                kg.add_edge(s, t, descriptions=[r.description], chunk_ids={cid})


def node_text(k):
    n = kg.nodes[k]
    return f"{n['name']} ({n['type']}): {' '.join(dict.fromkeys(n['descriptions']))}"


def edge_text(u, v):
    return f"{kg.nodes[u]['name']} -- {kg.nodes[v]['name']}: {' '.join(dict.fromkeys(kg.edges[u, v]['descriptions']))}"


communities = [c for c in nx.community.louvain_communities(kg, seed=0) if len(c) >= 2]
reporter = llm.with_structured_output(Report)
reports = []
for i, members in enumerate(communities):
    sub = kg.subgraph(members)
    body = "Entities:\n" + "\n".join(node_text(k) for k in members) + "\n\nRelationships:\n" + "\n".join(
        edge_text(u, v) for u, v in sub.edges())
    rep = reporter.invoke("Write a community report for this group of entities from a cloud host's handbook and "
                          "internal records.\n\n" + body, config={"callbacks": [graph_cb]})
    reports.append(Document(page_content=f"{rep.title}\n{rep.summary}", metadata={"community": i}))
entity_index = InMemoryVectorStore.from_documents(
    [Document(page_content=node_text(k), metadata={"entity": k}) for k in kg.nodes], embeddings)
report_index = InMemoryVectorStore.from_documents(reports, embeddings)
INDEX["GraphRAG"] = {"seconds": time.perf_counter() - start, "llm_tokens": tokens(graph_cb)}
print(kg.number_of_nodes(), "entities,", kg.number_of_edges(), "relationships,", len(reports), "community reports")
print(INDEX["GraphRAG"])

## GraphRAG: global and local search

Global search maps over the community reports closest to the question (up to `MAP_REPORTS`, which bounds the cost of a query), asks for the points each report adds with a 0 to 100 helpfulness score, and reduces the best points into one answer. Local search reads the nearest entities, their one-hop relationships and the chunks they came from.

In [ ]:
MAP_REPORTS = 8


class Points(BaseModel):
    points: list[str] = Field(description="Facts from the report that help answer the question; empty if none")
    score: int = Field(ge=0, le=100, description="How much this report helps answer the question")


mapper = llm.with_structured_output(Points)


def global_search(question: str) -> str:
    scored = []
    for rep in report_index.similarity_search(question, k=MAP_REPORTS):
        p = mapper.invoke(f"Question: {question}\n\nCommunity report:\n{rep.page_content}")
        if p.score > 0 and p.points:
            scored.append((p.score, p.points))
    scored.sort(key=lambda x: -x[0])
    notes = "\n".join(f"- ({s}) {pt}" for s, pts in scored for pt in pts)
    return answer_from("Points from community reports, with helpfulness scores:\n" + (notes or "(none)"), question)


def local_context(question: str, k: int = 5) -> str:
    ents, edges, cids = set(), set(), set()
    for hit in entity_index.similarity_search(question, k=k):
        e = hit.metadata["entity"]
        ents.add(e)
        cids |= entity_chunks[e]
        edges |= {tuple(sorted((e, n))) for n in kg.neighbors(e)}
    return ("Entities:\n" + "\n".join(node_text(e) for e in ents)
            + "\n\nRelationships:\n" + "\n".join(edge_text(u, v) for u, v in edges)
            + "\n\nSource passages:\n" + "\n---\n".join(chunks[c].page_content for c in sorted(cids)[:8]))


def local_search(question: str) -> str:
    return answer_from(local_context(question), question)


def graphrag(q: dict) -> str:
    return global_search(q["question"]) if q["set"] == "global" else local_search(q["question"])

## RAPTOR: index and search

The M42 build: Gaussian-mixture clusters of about five nodes, one summary per cluster, up to three levels, collapsed-tree retrieval.

In [ ]:
raptor_cb = UsageMetadataCallbackHandler()
start = time.perf_counter()
nodes, level = list(chunks), list(chunks)
for depth in range(1, 4):
    if len(level) <= 3:
        break
    vectors = np.array(embeddings.embed_documents([d.page_content for d in level]))
    labels = GaussianMixture(n_components=max(2, len(level) // 5), random_state=0).fit_predict(vectors)
    level = [Document(page_content=llm.invoke(
                 [SystemMessage("Summarise these passages. Keep every number, limit, name and date."),
                  HumanMessage("\n---\n".join(d.page_content for d, lab in zip(level, labels) if lab == c))],
                 config={"callbacks": [raptor_cb]}).content, metadata={"level": depth})
             for c in sorted(set(labels))]
    nodes += level
raptor_index = InMemoryVectorStore.from_documents(nodes, embeddings)
INDEX["RAPTOR"] = {"seconds": time.perf_counter() - start, "llm_tokens": tokens(raptor_cb)}
print(len(nodes) - len(chunks), "summaries |", INDEX["RAPTOR"])


def raptor_context(question: str, k: int = 6) -> str:
    return "\n---\n".join(d.page_content for d in raptor_index.similarity_search(question, k=k))


def raptor(q: dict) -> str:
    return answer_from(raptor_context(q["question"]), q["question"])

## Hybrid

One context with RAPTOR's top nodes (broad coverage) and GraphRAG's local context (entity paths). Its index is both builds, so its indexing cost is their sum.

In [ ]:
INDEX["hybrid"] = {k: INDEX["GraphRAG"][k] + INDEX["RAPTOR"][k] for k in ("seconds", "llm_tokens")}


def hybrid(q: dict) -> str:
    context = ("Summaries and passages:\n" + raptor_context(q["question"], k=4)
               + "\n\nKnowledge graph:\n" + local_context(q["question"], k=4))
    return answer_from(context, q["question"])

## Judge and run

In [ ]:
JUDGE_PROMPT = """You grade answers about a company's customer handbook and internal records.

Question: {question}
Reference answer (complete and correct): {reference}
Answer to grade: {answer}

Score 1 to 5: 5 = every point of the reference and no contradiction; 4 = most points, no contradiction;
3 = about half, or one minor error; 2 = few points or a clear error; 1 = misses the question or mostly wrong.
Give one sentence naming what is missing or wrong."""


class Verdict(BaseModel):
    score: int = Field(ge=1, le=5)
    reason: str


judge = llm.with_structured_output(Verdict)
print(JUDGE_PROMPT)

SYSTEMS = {"GraphRAG": graphrag, "RAPTOR": raptor, "hybrid": hybrid}
rows = []
for q in QUESTIONS:
    for name, system in SYSTEMS.items():
        start = time.perf_counter()
        answer = system(q)
        seconds = time.perf_counter() - start
        v = judge.invoke(JUDGE_PROMPT.format(question=q["question"], reference=q["reference_answer"], answer=answer))
        rows.append({"id": q["id"], "set": q["set"], "system": name, "score": v.score, "reason": v.reason,
                     "query_s": round(seconds, 2), "answer": answer})
    print(q["id"], [(r["system"], r["score"]) for r in rows[-3:]])

## Results

In [ ]:
import pandas as pd

df = pd.DataFrame(rows)
quality = df.pivot_table(index="system", columns="set", values="score", aggfunc="mean")
quality["median_query_s"] = df.groupby("system")["query_s"].median()
quality["index_s"] = [round(INDEX[s]["seconds"], 1) for s in quality.index]
quality["index_llm_tokens"] = [INDEX[s]["llm_tokens"] for s in quality.index]
print("model:", MODEL, "| embeddings:", type(embeddings).__name__, "| run:", time.strftime("%Y-%m-%d"))
print(quality.round(2).to_markdown())
print()
print(df.pivot(index="id", columns="system", values="score").to_markdown())

In [ ]:
for r in df[df["score"] < 4].itertuples():
    print(f"{r.id} {r.system} ({r.score}): {r.reason}")

## Recommendation

The cell below writes the one-page recommendation from the measured table, so its numbers are always this run's. It picks the best system per question set, weighs the gap against indexing cost, and states when each system is the right choice.

In [ ]:
def recommendation(quality: pd.DataFrame) -> str:
    best_global = quality["global"].idxmax()
    best_multi = quality["multi_hop"].idxmax()
    cheapest = quality["index_llm_tokens"].idxmin()
    q = quality.round(2)
    lines = [
        f"# Recommendation: retrieval for global and multi-hop questions (run {time.strftime('%Y-%m-%d')}, {MODEL})",
        "",
        f"Corpus: {len(pages)} documents, {len(chunks)} chunks. Questions: "
        f"{(df['set'] == 'global').sum() // 3} global, {(df['set'] == 'multi_hop').sum() // 3} multi-hop. "
        "Scores are an LLM judge's 1 to 5 against written reference answers.",
        "",
        "| System | Global | Multi-hop | Index tokens | Index seconds | Median query seconds |",
        "|---|---|---|---|---|---|",
        *[f"| {s} | {q.loc[s, 'global']} | {q.loc[s, 'multi_hop']} | {int(q.loc[s, 'index_llm_tokens'])} | "
          f"{q.loc[s, 'index_s']} | {q.loc[s, 'median_query_s']} |" for s in q.index],
        "",
        f"- Global questions: {best_global} scored highest ({q.loc[best_global, 'global']}).",
        f"- Multi-hop questions: {best_multi} scored highest ({q.loc[best_multi, 'multi_hop']}).",
        f"- Cheapest to index: {cheapest} ({int(q.loc[cheapest, 'index_llm_tokens'])} model tokens).",
        "",
    ]
    if best_global == best_multi:
        choice = best_global
        lines.append(f"Recommendation: use {choice} for both question types.")
    else:
        choice = None
        lines.append(f"Recommendation: route by question type, {best_global} for global questions and "
                     f"{best_multi} for multi-hop ones, or use the hybrid if one index is simpler to run.")
    for s in q.index:
        gap = max(q["global"].max() - q.loc[s, "global"], q["multi_hop"].max() - q.loc[s, "multi_hop"])
        if s != choice and gap < 0.5 and q.loc[s, "index_llm_tokens"] < q.loc[choice or best_global, "index_llm_tokens"]:
            lines.append(f"- {s} is within half a point of the best on both sets and cheaper to index; "
                         "prefer it if the corpus changes often, since every change means re-indexing.")
    lines += [
        "",
        "When each fits:",
        "- GraphRAG: questions about the whole corpus (global search over community reports) and about chains of "
        "entities (local search). Its index costs one extraction call per chunk plus one report per community, and "
        "every corpus change can move community boundaries.",
        "- RAPTOR: broad questions whose answer a cluster summary can hold. Cheaper to index than GraphRAG; summaries "
        "must be rebuilt for the clusters a change touches.",
        "- Hybrid: when both question types arrive and routing is unreliable. It pays both indexing costs.",
        "",
        "Caveats: one run, one model, 15 questions, and a judge with its own errors. Read the judge's reasons above "
        "before acting on a gap under one point, and re-run on your own questions before replacing flat retrieval.",
    ]
    return "\n".join(lines)


print(recommendation(quality))